* 임베딩 공간의 정확한 차원은 실무자의 선택에 달려있음
* 최적의 임베딩 차원은 실험을 통해서 찾아내야함
* 그러나 기존 연구를 토대로
* - 1) 고유한 카테고리형 원소 총수의 네 제곱근
* - 2) 임베딩 차원이 고유한 카테고리형 원소 총수 제곱근의 약 1.6배
* - ex) 625개의 고유한 값이 있는 특징을 인코딩한다고 가정, 1번 - 5, 2번 - 40
* - 하이퍼파라미터 튜닝을 통해 임베딩 차원을 찾는다면 이 범위내에서 찾아보는 것이 의미 있을것

In [1]:
import pandas as pd

In [11]:
DATASET = "./data/titles_full.csv"
COLUMNS = ["title", "source"]

titles_df = pd.read_csv(DATASET, header=None, names=COLUMNS)
titles_df.head()

,title,source
0,holy cash cow batman - content is back,nytimes
1,show hn a simple and configurable deployment ...,github
2,show hn neural turing machine in pure numpy. ...,github
3,close look at a flu outbreak upends some commo...,nytimes
4,lambdalite a functional relational lisp data...,github


In [8]:
%pip install tensorflow_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 4.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [tensorflow_hub]
Note: you may need to restart the kernel to use updated packages.


In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow import keras
from tensorflow.keras import callbacks, layers, models, utils
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow_hub import KerasLayer

/opt/homebrew/lib/python3.12/site-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


In [ ]:
tokenizer = Tokenizer()

# 단어 집합 구축
# titles 에서 각 단어를 인덱스에 매칭하는 조회 테이블이 생성됨
tokenizer.fit_on_texts(titles_df.title) 
print(tokenizer.index_word) 
integerized_titles = tokenizer.texts_to_sequences(titles_df.title) # 텍스트를 정수 시퀀스로 변환

{1: 'the', 2: 'a', 3: 'to', 4: 'for', 5: 'in', 6: 'of', 7: 'and', 8: 's', 9: 'on', 10: 'with', 11: 'show', 12: 'hn', 13: 'is', 14: 'google', 15: 'your', 16: 'new', 17: 'from', 18: 'js', 19: 'an', 20: 'facebook', 21: 'you', 22: 'app', 23: 'at', 24: 'it', 25: 'web', 26: 'how', 27: 'by', 28: 'github', 29: 'that', 30: 'its', 31: 'as', 32: 'data', 33: 'javascript', 34: 'apple', 35: 'open', 36: 'twitter', 37: 't', 38: 'python', 39: 'be', 40: '1', 41: 'up', 42: 'now', 43: 'are', 44: 'library', 45: 'node', 46: 'source', 47: 'raises', 48: 'million', 49: 'simple', 50: 'mobile', 51: 'launches', 52: '2', 53: 'more', 54: 'go', 55: 'i', 56: 'using', 57: 'c', 58: 'can', 59: 'android', 60: 'why', 61: 'what', 62: 'startup', 63: 'will', 64: 'into', 65: 'not', 66: 'api', 67: 'like', 68: 'apps', 69: 'based', 70: 'microsoft', 71: '0', 72: 'yc', 73: 'code', 74: 'ios', 75: 'social', 76: 'out', 77: 'iphone', 78: 'all', 79: 'time', 80: 'ruby', 81: 'search', 82: 'tech', 83: 'framework', 84: 'one', 85: 'video', 

In [15]:
integerized_titles[:5]

[[6117, 560, 8577, 13948, 302, 13, 172],
 [11, 12, 2, 49, 7, 3838, 1322, 91, 4, 28, 482],
 [11, 12, 1501, 2812, 322, 5, 589, 7337, 5458, 78, 108, 1989, 17, 1139],
 [1030, 316, 23, 2, 3718, 7338, 13949, 214, 715, 4581],
 [23705, 2, 624, 3605, 529, 290, 5, 2139, 517, 6, 715, 529]]

In [22]:
VOCAB_SIZE = len(tokenizer.index_word)
print(VOCAB_SIZE)

47271


In [23]:
DATASET_SIZE = tokenizer.document_count
print(DATASET_SIZE)

96203


In [24]:
MAX_LEN = max(len(sequence) for sequence in integerized_titles)
print(MAX_LEN)

26


In [ ]:
# preprocess Data
# 모델의 첫번째 레이어(임베딩 레이어)는 입력되는 차원이 일정해야하기 때문에
# 시퀀스의 길이를 동일하게 맞춰주는 패딩 작업이 필요함
# 시퀀스의 길이를 최대 길이를 기준으로 padding을 진행

def create_sequences(texts, max_len=MAX_LEN):
    sequences = tokenizer.texts_to_sequences(texts)

     # post : 시퀀스의 뒤에 패딩을 추가
    padded_sequences = pad_sequences(sequences, 
                                     max_len,
                                     padding="post")
    return padded_sequences



In [26]:
sample_titles = create_sequences(["holy cash cow  batman - content is back",
                                 "close look at a flu outbreak upends some common wisdom"])
sample_titles

array([[ 6117,   560,  8577, 13948,   302,    13,   172,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0],
       [ 1030,   316,    23,     2,  3718,  7338, 13949,   214,   715,
         4581,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0]],
      dtype=int32)

In [27]:
CLASSES = {
    "github": 0,
    "nytimes": 1,
    "techcrunch": 2,
}
N_CLASSES = len(CLASSES)

In [28]:
def encode_labels(sources):
    classes = [CLASSES[source] for source in sources]
    # utils.to_categorical : 정수로 인코딩된 클래스 레이블을 원-핫 인코딩으로 변환
    one_hots = utils.to_categorical(classes) 
    return one_hots

In [30]:
N_TRAIN = int(0.8 * DATASET_SIZE)

# 훈련 데이터와 검증 데이터로 분할
# 0.8 : 0.2 비율로 분할
titles_train = titles_df.title[:N_TRAIN]
sources_train = titles_df.source[:N_TRAIN]

titles_valid = titles_df.title[N_TRAIN:]
sources_valid = titles_df.source[N_TRAIN:]

In [31]:
sources_train.value_counts()

source
github        29175
techcrunch    24784
nytimes       23003
Name: count, dtype: int64

In [32]:
X_train, Y_train = create_sequences(titles_train), encode_labels(sources_train)
X_valid, Y_valid = create_sequences(titles_valid), encode_labels(sources_valid)

In [33]:
X_train[:3], Y_train[:3]

(array([[ 6117,   560,  8577, 13948,   302,    13,   172,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0],
        [   11,    12,     2,    49,     7,  3838,  1322,    91,     4,
            28,   482,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0],
        [   11,    12,  1501,  2812,   322,     5,   589,  7337,  5458,
            78,   108,  1989,    17,  1139,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0]],
       dtype=int32),
 array([[0., 1., 0.],
        [1., 0., 0.],
        [1., 0., 0.]]))

In [34]:
import tensorflow as tf

In [35]:
# BUILD MODEL(DNN)

def build_dnn_model(embed_dim):
    model = models.Sequential([
        # index -> embedding vector
        layers.Embedding(VOCAB_SIZE + 1, # 사용할 단어의 총 개수, +1 은 패딩 토큰을 위한 공간
                         embed_dim, # 한 단어를 몇 차원의 벡터로 표현할 것인지
                         input_shape=[MAX_LEN]), # 모델에 들어오는 문장 길이

        # embedding 층을 통과하면 (문장길이, 벡터 차원) 형태의 결과가 나옴
        # 이를 단일 벡터로 압축
        # 각 단어의 임베딩 벡터를 평균하여 문장 전체를 하나의 벡터로 표현
        # reduce_mean : 텐서의 특정 차원에 대한 평균을 계산하는 함수
        # axis=1 : 문장 길이 방향으로 평균을 내서 2차원 데이터를 1차원으로
        # (Global Average Pooling) 느낌
        layers.Lambda(lambda x: tf.reduce_mean(x, axis=1)),

        # 추출된 특징을 바탕으로 최종적으로 어떤 클래스에 속할지 결정
        # N_CLASSES : 클래스의 개수
        layers.Dense(N_CLASSES, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"] # 학습하는 동안 사람이 확인하기 위한평가지표, 현재는 "정확도"
    )
    return model

In [36]:
Y_train.shape

(76962, 3)

In [ ]:
%%time 
# 주피터 노트북 매직 커맨드, 셀이 실행되는데 걸린 총 시간을 측정, 출력


tf.random.set_seed(42)

BATCH_SIZE = 300
EPOCHS = 10
EMBED_DIM = 10
PATIENCE = 0 # EarlyStopping 콜백에서 모니터링하는 지표가 개선되지 않는 에포크 수를 지정, 0으로 설정하면 개선이 없으면 바로 학습 중단

dnn_model = build_dnn_model(EMBED_DIM)
dnn_history = dnn_model.fit(
    X_train, Y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_valid, Y_valid),
    # early stopping : 검증 손실이 개선되지 않으면 학습을 중단하는 콜백, 모델이 과적합되는 것을 방지하기 위해 사용
    # - monitor : 관찰할 지표(예: 'val_loss')
    # - patience : 개선이 없을 때 기다리는 에포크 수(0으로 설정하면 개선이 없으면 바로 중단)
    # - restore_best_weights : 학습이 중단된 시점에서 가장 좋은 모델의 가중치를 복원할지 여부(True/False)
    # TensorBoard : 텐서보드 로그를 저장하는 콜백, 모델의 학습 과정을 시각화하는 데 사용
    callbacks=[callbacks.EarlyStopping(patience=PATIENCE),
               callbacks.TensorBoard(log_dir='./logs')],
)

pd.DataFrame(dnn_history.history)[['loss', 'val_loss']].plot()
pd.DataFrame(dnn_history.history)[['accuracy', 'val_accuracy']].plot()

dnn_model.summary()

Epoch 1/10


/opt/homebrew/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:103: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
2026-02-20 14:48:22.790433: I tensorflow/core/common_runtime/eager/kernel_and_device.cc:100] Ignoring error status when releasing multi-device function handle INTERNAL: Inconsistent FunctionLibraryRuntime. Expected to find an item for handle 41 but found none
2026-02-20 14:48:22.790455: I tensorflow/core/common_runtime/eager/kernel_and_device.cc:100] Ignoring error status when releasing multi-device function handle INTERNAL: Inconsistent FunctionLibraryRuntime. Expected to find an item for handle 37 but found none
2026-02-20 14:48:22.790461: I tensorflow/core/common_runtime/eager/kernel_and_device.cc:100] Ignoring error status when releasing multi-device function handle INTERNAL: Inconsistent Fu

In [ ]:
# 전이학습 코드
NNLM = "https://tfhub.dev/google/nnlm-en-dim50/2"

nnlm_module = KerasLayer(
    handle=NNLM,         # 불러올 모델의 경로(주소)
    output_shape=[50],   # 출력될 벡터의 크기 (50차원)
    input_shape=[],      # 입력 데이터의 형태 (보통 1차원 문자열 배열이 들어오므로 비워둠)
    dtype=tf.string,     # 입력받을 데이터의 타입 (텍스트이므로 string)

    # 내 데이터에 맞게 가중치를 미세 조정(Fine-tuning)할지 여부
    # True : 내가 가진 특정 데이터셋의 특성에 맞춰 기존 모델의 수치를 조금씩 업데이트
    # False : 모델을 그대로(Freeze) 사용
    trainable=True      
)
